# RF / XGBoost：时间特征与诊断 embedding 消融

本次只补代码，没有运行真实患者实验。所有参数、开发集和外层划分保持 balanced 原定义。

## 比较设计

- static_rf → temporal_rf；static_xgb → temporal_xgb：检验时间特征在不同模型下的贡献。
- static_rf → burden_rf → embedding_rf；XGBoost 同理：先控制诊断数量，再判断潜在表示的增益。
- embedding 另外对比同模型 static，报告整个诊断扩展的总增益。
- 仍使用相同 LASSO 数值筛选，不把“换学习器”和“换筛选方法”同时混入一个比较。

此入口只拟合独立基模型，不进行 stacking 内部交叉拟合。已有 stacking 预测可复用作历史汇总，不会因此重新训练 stacking。

## 诊断 embedding 的实际流程

1. 仅取索引住院以前已经出院的住院：`prior.dischtime < index.admittime`，不使用本次住院出院诊断或未来住院。
2. ICD 码去小数点、转大写，ICD-9/10 分别标记。每个确切码与前三字符类目生成 token。
   例如 ICD-10 `E11.9` 变成 `v10_E119` 和 `c10_E11`；ICD-9 用 `v9_` / `c9_`，不会混作同一码。
3. 每人历史诊断 token 去重后形成一个文档。因 token 去重，TF 的 sublinear_tf 在这里通常不体现重复次数；诊断频次另由 burden 表示。
4. 在训练折拟合 TF-IDF，`min_df=10`（至少10个训练患者文档中出现）；验证集只使用训练词表。
5. 训练折拟合 TruncatedSVD，最多32维；验证集只 transform，不参与词表、IDF 或 SVD 学习。
6. 数值静态/时间特征经折内缺失率筛选、插补和 LASSO；四个 burden 指标及 embedding 在 LASSO 后直接追加。
   树模型没有额外 StandardScaler；前三个 burden 计数做 log1p。这些均继承现有 Pipeline。

这是一种结构化诊断的统计表示，不是临床文本或医学语言模型。SVD 各维不能直接命名为某一种疾病，也不保证维度数增加就提升预测。
没有历史诊断的患者保留为空文档；空词表或单词表有原有回退逻辑。ICD-10 取前三字符是代码分组方式，并非完整临床疾病层级映射。

## 可复现与缓存

种子42–46，外层10折；LASSO内部5折/30λ，1-SE；缺失率阈值20%，沿用原静态白细胞分类豁免。
RF/XGBoost 参数继续读取原 config.yaml，未调参。固定留出集标签不参与。

`--reuse-from ebf3f368bb8beb5c` 会验证旧版训练函数、配置、数据、软件环境和每折索引，再复制已有预测。
旧结果保留，新实验写入新签名；若不匹配则报错，不绕过校验。结果中的 Holm 校正随当前比较家族更新。

终端从原扩展目录运行：
```bash
bash balanced/run_extensions.sh full --experiments temporal_rf,temporal_xgb,embedding_rf,embedding_xgb --reuse-from ebf3f368bb8beb5c
```
命令末尾加 `--check-only` 只校验，不训练。也可以把四个实验分成时间与 embedding 两次运行。


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dn_extensions/settings.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('请在 balanced 目录或其 notebooks 子目录启动。')
sys.path.insert(0, str(ROOT))
print('项目目录：', ROOT)


In [ ]:
# 默认不训练；由你确认后手动设置 RUN_EXPERIMENT=True。
RUN_EXPERIMENT = False
EXPERIMENTS = ['temporal_rf', 'temporal_xgb', 'embedding_rf', 'embedding_xgb']
REUSE_FROM = 'ebf3f368bb8beb5c'
if RUN_EXPERIMENT:
    from dn_extensions.evaluation import run_evaluation
    from dn_extensions.settings import profile_options
    output, summary, comparisons = run_evaluation(
        profile_options('full'), EXPERIMENTS, reuse_from=REUSE_FROM)
    display(summary)
    display(comparisons)
else:
    print('未启动训练；可以使用上面的 Bash 命令运行。')


In [ ]:
# 只查看已有结果，不重新训练。
latest = ROOT / 'results/full/latest.txt'
if latest.exists():
    folder = latest.parent / latest.read_text().strip()
    table = pd.read_csv(folder / 'summary.csv')
    names = EXPERIMENTS + ['static_rf', 'static_xgb', 'burden_rf', 'burden_xgb']
    display(table[table.experiment.isin(names)])
    print('详细结果：', folder / 'report.md')
else:
    print('尚无 full 结果。')


## 实验状态与结论（自动更新）

运行模式：`full`；方法：balanced 固定参数；结果目录：`results/full/27464ba98305edbc`。

LASSO 5折/30λ、stacking 3折；静态与扩展使用同一简化方法。严格版的部分折没有混入。

固定留出集未参与本实验。树模型扩展不需要 stacking 内部交叉拟合；原固定参数及训练折内预处理不变。

Holm 校正覆盖当前已有配对比较；后续增加实验时，校正 p 值可能变化。

```text

   experiment  n_outer_folds  AUROC_fold_mean  AUROC_fold_sd  AUPRC_fold_mean  Brier_fold_mean  calibration_slope_fold_mean  seconds_total  AUROC_pooled_mean  AUPRC_pooled_mean  Brier_pooled_mean  cal_intercept_pooled_mean  cal_slope_pooled_mean
    burden_rf             50          0.79724        0.02871          0.60343          0.15232                      1.22278       28.27988            0.79581            0.58966            0.15232                    0.00337                1.18404
   burden_xgb             50          0.79396        0.03022          0.60583          0.15265                      0.93979        6.26590            0.79309            0.59511            0.15264                    0.00653                0.91576
 embedding_rf             50          0.79965        0.02798          0.60840          0.15269                      1.39541       32.57728            0.79825            0.59542            0.15268                    0.00490                1.35224
embedding_xgb             50          0.79429        0.02760          0.60649          0.15232                      0.97897        9.29661            0.79331            0.59430            0.15231                   -0.00756                0.95188
    static_rf             50          0.79798        0.02809          0.60618          0.15177                      1.19841       15.22184            0.79671            0.59435            0.15177                    0.00179                1.16211
   static_xgb             50          0.79504        0.02965          0.60572          0.15250                      0.94083        4.61352            0.79408            0.59662            0.15249                    0.00766                0.91690
  temporal_rf             50          0.79741        0.02891          0.60594          0.15212                      1.24090       28.89751            0.79605            0.59470            0.15211                    0.00148                1.20217
 temporal_xgb             50          0.79284        0.02796          0.60391          0.15300                      0.94159        8.29193            0.79177            0.59437            0.15299                    0.01106                0.91792

```

配对差异：

```text

   experiment  reference metric  mean_diff  sd_diff  se_corrected       lo      hi        t  df  p_value  k  r  p_holm_current_family
    static_rf  static_lr  AUROC   -0.00130  0.01457       0.00528 -0.01190 0.00931 -0.24552  49  0.80708 10  5                    1.0
    static_rf  static_lr  AUPRC   -0.00122  0.02900       0.01050 -0.02232 0.01989 -0.11569  49  0.90837 10  5                    1.0
    static_rf  static_lr  Brier    0.00085  0.00469       0.00170 -0.00256 0.00427  0.50207  49  0.61787 10  5                    1.0
   static_xgb  static_lr  AUROC   -0.00423  0.01409       0.00510 -0.01449 0.00602 -0.82968  49  0.41074 10  5                    1.0
   static_xgb  static_lr  AUPRC   -0.00167  0.02894       0.01048 -0.02273 0.01938 -0.15972  49  0.87376 10  5                    1.0
   static_xgb  static_lr  Brier    0.00158  0.00519       0.00188 -0.00220 0.00536  0.83926  49  0.40540 10  5                    1.0
  temporal_rf  static_rf  AUROC   -0.00057  0.00733       0.00265 -0.00591 0.00476 -0.21556  49  0.83022 10  5                    1.0
  temporal_rf  static_rf  AUPRC   -0.00024  0.01373       0.00497 -0.01023 0.00975 -0.04896  49  0.96115 10  5                    1.0
  temporal_rf  static_rf  Brier    0.00034  0.00196       0.00071 -0.00108 0.00177  0.48410  49  0.63047 10  5                    1.0
 temporal_xgb static_xgb  AUROC   -0.00220  0.01022       0.00370 -0.00964 0.00523 -0.59579  49  0.55405 10  5                    1.0
 temporal_xgb static_xgb  AUPRC   -0.00181  0.02169       0.00785 -0.01759 0.01397 -0.23090  49  0.81836 10  5                    1.0
 temporal_xgb static_xgb  Brier    0.00050  0.00389       0.00141 -0.00233 0.00334  0.35547  49  0.72376 10  5                    1.0
    burden_rf  static_rf  AUROC   -0.00074  0.00616       0.00223 -0.00522 0.00374 -0.33253  49  0.74091 10  5                    1.0
    burden_rf  static_rf  AUPRC   -0.00275  0.01287       0.00466 -0.01212 0.00661 -0.59075  49  0.55740 10  5                    1.0
    burden_rf  static_rf  Brier    0.00055  0.00173       0.00063 -0.00071 0.00181  0.87819  49  0.38412 10  5                    1.0
   burden_xgb static_xgb  AUROC   -0.00108  0.00676       0.00245 -0.00600 0.00384 -0.44121  49  0.66100 10  5                    1.0
   burden_xgb static_xgb  AUPRC    0.00010  0.01749       0.00633 -0.01263 0.01283  0.01602  49  0.98728 10  5                    1.0
   burden_xgb static_xgb  Brier    0.00015  0.00214       0.00077 -0.00140 0.00171  0.19740  49  0.84433 10  5                    1.0
 embedding_rf  burden_rf  AUROC    0.00241  0.00885       0.00320 -0.00403 0.00885  0.75282  49  0.45516 10  5                    1.0
 embedding_rf  burden_rf  AUPRC    0.00497  0.01607       0.00582 -0.00673 0.01666  0.85397  49  0.39728 10  5                    1.0
 embedding_rf  burden_rf  Brier    0.00037  0.00239       0.00087 -0.00137 0.00211  0.42331  49  0.67392 10  5                    1.0
embedding_xgb burden_xgb  AUROC    0.00033  0.01064       0.00385 -0.00741 0.00807  0.08550  49  0.93221 10  5                    1.0
embedding_xgb burden_xgb  AUPRC    0.00067  0.02335       0.00846 -0.01632 0.01766  0.07910  49  0.93728 10  5                    1.0
embedding_xgb burden_xgb  Brier   -0.00033  0.00395       0.00143 -0.00321 0.00254 -0.23263  49  0.81702 10  5                    1.0
 embedding_rf  static_rf  AUROC    0.00167  0.00963       0.00349 -0.00534 0.00868  0.47943  49  0.63377 10  5                    1.0
 embedding_rf  static_rf  AUPRC    0.00222  0.01939       0.00702 -0.01189 0.01632  0.31582  49  0.75348 10  5                    1.0
 embedding_rf  static_rf  Brier    0.00092  0.00260       0.00094 -0.00098 0.00281  0.97278  49  0.33544 10  5                    1.0
embedding_xgb static_xgb  AUROC   -0.00075  0.00988       0.00358 -0.00794 0.00644 -0.20974  49  0.83474 10  5                    1.0
embedding_xgb static_xgb  AUPRC    0.00077  0.02388       0.00865 -0.01661 0.01815  0.08909  49  0.92938 10  5                    1.0
embedding_xgb static_xgb  Brier   -0.00018  0.00389       0.00141 -0.00301 0.00265 -0.12793  49  0.89873 10  5                    1.0

```

- `static_rf` 相对 `static_lr`：ΔAUROC=-0.0013，校正95% CI [-0.0119, 0.0093]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `static_xgb` 相对 `static_lr`：ΔAUROC=-0.0042，校正95% CI [-0.0145, 0.0060]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `temporal_rf` 相对 `static_rf`：ΔAUROC=-0.0006，校正95% CI [-0.0059, 0.0048]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `temporal_xgb` 相对 `static_xgb`：ΔAUROC=-0.0022，校正95% CI [-0.0096, 0.0052]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `burden_rf` 相对 `static_rf`：ΔAUROC=-0.0007，校正95% CI [-0.0052, 0.0037]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `burden_xgb` 相对 `static_xgb`：ΔAUROC=-0.0011，校正95% CI [-0.0060, 0.0038]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_rf` 相对 `burden_rf`：ΔAUROC=0.0024，校正95% CI [-0.0040, 0.0089]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_xgb` 相对 `burden_xgb`：ΔAUROC=0.0003，校正95% CI [-0.0074, 0.0081]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_rf` 相对 `static_rf`：ΔAUROC=0.0017，校正95% CI [-0.0053, 0.0087]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。

- `embedding_xgb` 相对 `static_xgb`：ΔAUROC=-0.0008，校正95% CI [-0.0079, 0.0064]；尚无经本轮多重比较校正确认的差异证据。还需查看 Holm 校正 p 值、AUPRC、校准和 DCA。